# 06 · Pandas drills — groupby, merge, reshape, dates

**Datasets:** `data/orders.csv` (5,000 orders) + `data/customers.csv` (400 customers).
**Covers:** the wrangling muscle memory — groupby, merge, pivot, window functions,
datetimes, ranking.
**Time yourself:** ~30 minutes, and try to keep each answer to one chained expression.

These are the "quick, get me X" questions that fill the first 10 minutes of a live
coding round. There is no modeling here — it's pure speed and fluency. The point is
to not have to think about `groupby` syntax while you're also thinking about the problem.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

orders = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
customers = pd.read_csv('data/customers.csv', parse_dates=['signup_date'])

print('orders   ', orders.shape)
print('customers', customers.shape)
display(orders.head(3))
display(customers.head(3))

---

## Part A — Groupby

### Q1. Total revenue and number of orders per `category`, sorted by revenue descending.

<details><summary>hint</summary>

Named aggregation: `.agg(new_name=('col', 'func'))`.

</details>

In [ ]:
# your code here

### Q2. Average order value per `channel` **and** `category` together. Return it as a wide
table with channels as rows and categories as columns.

<details><summary>hint</summary>

`pivot_table` — or `groupby([...]).mean().unstack()`, which is the same thing the long way round.

</details>

In [ ]:
# your code here

### Q3. For each `category`, compute total revenue, mean revenue, median revenue, and the
number of *distinct customers* who bought from it. Four aggregations, one call.

<details><summary>hint</summary>

`'nunique'` is the aggregation you want for distinct counts.

</details>

In [ ]:
# your code here

### Q4. `revenue` has some NaNs. How many, and does `groupby().sum()` include or ignore them?
Prove it, then recompute `revenue` from `quantity * unit_price` where it's missing.

<details><summary>hint</summary>

The difference between `.count()` and `.size()` on a groupby is exactly the NaN handling. That's the whole answer.

</details>

In [ ]:
# your code here

---

## Part B — Merge

### Q5. Join `orders` to `customers` to get the country on every order. Use an inner join,
then check: did you lose rows? If so, why — and is that OK?

<details><summary>hint</summary>

Always compare `len()` before and after a join. `~df['k'].isin(other['k'])` finds the rows that fell out.

</details>

In [ ]:
# your code here

### Q6. Redo it as a left join and use the indicator to quantify the mismatch precisely.
Which join would you use in a revenue report, and why?

<details><summary>hint</summary>

`how='left', indicator=True` adds a `_merge` column with values `both` / `left_only`. It's the fastest join sanity check there is.

</details>

In [ ]:
# your code here

### Q7. Using the merged data: total revenue per `country`, and the average revenue per customer in each country.

<details><summary>hint</summary>

`.assign(new=lambda d: ...)` lets you derive a column from the aggregation you just built, without breaking the chain.

</details>

In [ ]:
# your code here

---

## Part C — Dates

### Q8. Monthly total revenue. Then find the single best month.

<details><summary>hint</summary>

`groupby(df['order_date'].dt.to_period('M'))`, or `.set_index('order_date').resample(...)` — but check your pandas version for the resample alias ('M' vs 'ME').

</details>

In [ ]:
# your code here

### Q9. Revenue by day of week. Is the weekend different? Show the day names, in calendar
order, not alphabetical.

<details><summary>hint</summary>

`.dt.day_name()` gives the string. `.reindex(list_of_days)` forces calendar order — otherwise you get Friday first.

</details>

In [ ]:
# your code here

### Q10. For each customer, how many days passed between their signup and their **first**
order? Show the distribution.

<details><summary>hint</summary>

Subtracting two datetime columns gives a timedelta; `.dt.days` turns it into an integer.

</details>

In [ ]:
# your code here

### Q11. Compute a 3-month rolling average of monthly revenue.

<details><summary>hint</summary>

`.rolling(3).mean()`. The first two rows are NaN by design — there's no window yet.

</details>

In [ ]:
# your code here

---

## Part D — Window functions & ranking

### Q12. What share of each category's revenue does each `channel` contribute? You need the
group total broadcast back onto every row — that's `transform`, not `agg`.

<details><summary>hint</summary>

`transform('sum')` returns a Series the same length as the input, so you can divide by it. `agg` collapses; `transform` broadcasts.

</details>

In [ ]:
# your code here

### Q13. Find the **top 2 customers by total revenue within each country**. This is the classic
'top-N per group' question — they will ask it.

<details><summary>hint</summary>

Sort descending, then `.groupby(...).head(2)` — head on a groupby is per-group. Know the `.rank()` version too; it's what they mean if they say 'window function'.

</details>

In [ ]:
# your code here

### Q14. Flag each customer's orders with a running cumulative revenue total, ordered by date.
Show it for one customer to prove it works.

<details><summary>hint</summary>

Sort by the group key and the date FIRST, then `groupby(...).cumsum()`. Without the sort your cumulative total is in arbitrary order and silently wrong.

</details>

In [ ]:
# your code here

### Q15. For each customer, compute the number of days since their **previous** order
(`NaN` for their first). Then: what's the median gap between orders?

<details><summary>hint</summary>

`.shift(1)` within a groupby is the lag. Same rule as cumsum: sort first.

</details>

In [ ]:
# your code here

---

## Part E — Reshape

### Q16. Build a customer-level feature table (the kind you'd feed a churn model): one row per
customer with total revenue, order count, average order value, distinct categories,
first and last order date, and their country and segment.

<details><summary>hint</summary>

One `groupby().agg()` with named aggregations, then `.join()` the customer attributes on the index. This shape — one row per entity — is what every tabular model wants.

</details>

In [ ]:
# your code here

### Q17. Add one column per category holding that customer's spend in it (`spend_electronics`,
`spend_books`, …). This is long → wide.

<details><summary>hint</summary>

`pivot_table` with `fill_value=0` — a customer who never bought books should get 0, not NaN. `.add_prefix()` names the columns.

</details>

In [ ]:
# your code here

### Q18. Reverse it: take the `spend_*` columns back to long format with one row per
(customer, category) — dropping the zeros.

<details><summary>hint</summary>

`.melt(id_vars=..., value_vars=...)` is the inverse of pivot. `str.removeprefix` strips the prefix back off.

</details>

In [ ]:
# your code here

---

## Debrief — the five that come up every time

1. `groupby().agg(name=('col', 'fn'))` — named aggregation. Know it cold.
2. `transform` vs `agg`. Broadcast vs collapse. Any "% of group total" question is
   `transform`.
3. **Top-N per group**: `sort_values().groupby().head(n)`, and the `.rank()` variant.
4. **Check every merge.** Row count before, row count after, `indicator=True` if they
   differ. An inner join deleting 3% of your revenue in silence is a real outage.
5. `shift` / `cumsum` / `rolling` — always sort by the group and time key first.

**The sneaky one:** `.count()` vs `.size()` on a groupby differ by exactly the NaNs.